In [1]:
import sys
from pathlib import Path
raiz = Path.cwd()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
from src.cuantizacion import (ajustar_cuantizador, transformar, reconstruir,
                              calcular_error, REPRESENTANTES)
DIR_TABLAS = raiz / "resultados" / "actividad3" / "tablas"
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

In [2]:
rng = np.random.default_rng(2)
x_ceros = np.concatenate([np.zeros(7000), rng.exponential(size=3000) + 0.1])
rng.shuffle(x_ceros)

mc = ajustar_cuantizador(x_ceros, "cuantiles", 10)
print("bins solicitados:", mc.n_bins_solicitado, "| bins reales:", mc.n_bins_real)
print("límites:", np.round(mc.limites, 3))
print("observaciones por bin:", np.bincount(transformar(x_ceros, mc)))

bins solicitados: 10 | bins reales: 4
límites: [0.    0.03  0.498 1.179 7.94 ]
observaciones por bin: [7000 1000 1000 1000]


In [3]:
mv = ajustar_cuantizador([0, 1, 2, 100], "uniforme", 4)
print("límites:", mv.limites)
for r in REPRESENTANTES:
    print(f"{r:12s}", mv.representantes[r])

límites: [  0.  25.  50.  75. 100.]
punto_medio  [12.5 37.5 62.5 87.5]
media        [  1.   37.5  62.5 100. ]
mediana      [  1.   37.5  62.5 100. ]


In [4]:
ing = pd.read_parquet(raiz / "data" / "dataset_1M.parquet", columns=["ingresos"])["ingresos"]
ing = ing.sample(100_000, random_state=0).to_numpy()

filas = []
for estrategia in ("uniforme", "cuantiles"):
    mi = ajustar_cuantizador(ing, estrategia, 10, columna="ingresos")
    cod = transformar(ing, mi)
    print(estrategia, "| bins reales:", mi.n_bins_real,
          "| observaciones por bin:", np.bincount(cod, minlength=mi.n_bins_real))
    for r in REPRESENTANTES:
        e = calcular_error(ing, reconstruir(cod, mi, r))
        filas.append({"estrategia": estrategia, "representante": r, **e})
pd.DataFrame(filas)

uniforme | bins reales: 10 | observaciones por bin: [36938 45238 13174  3368   925   227    79    27    16     8]
cuantiles | bins reales: 10 | observaciones por bin: [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000]


,estrategia,representante,mae,rmse,error_max,n_validos
0,uniforme,punto_medio,4002.900496,4638.638067,8103.108899,100000
1,uniforme,media,3433.518494,4089.654178,11174.233778,100000
2,uniforme,mediana,3410.500540,4127.048945,11546.755640,100000
3,cuantiles,punto_medio,5938.092873,16075.795732,61361.531966,100000
4,cuantiles,media,1825.337689,4340.625992,110252.834239,100000
5,cuantiles,mediana,1760.636475,4534.172646,114378.981116,100000
